In [ ]:
#!pip install --upgrade langchain langchain-community langgraph openai langchain-openai

In [1]:
from langgraph.graph import StateGraph, START
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import MessagesState
from IPython.display import Image, display
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from typing import Annotated
from typing_extensions import TypedDict

ImportError: cannot import name 'ToolNode' from 'langgraph.prebuilt' (unknown location)

In [ ]:
class State(TypedDict):
  messages: Annotated[list, add_messages]

In [ ]:
!pip install duckduckgo-search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults, DuckDuckGoSearchRun

In [ ]:
search = DuckDuckGoSearchResults()
search.invoke("Obama's first name?")

In [ ]:
search = DuckDuckGoSearchRun()
search.invoke("Obama's first name?")

In [ ]:
search.invoke("who is Narendra Modi?")

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

def search_duckduckgo(query: str):
  search = DuckDuckGoSearchRun()
  return search.invoke(query)

In [ ]:
result = search_duckduckgo("What are AI agent?")
print(result)

In [ ]:
result = search_duckduckgo("What is Newton's Third Law?")
print(result)

In [ ]:
from langchain_openai import ChatOpenAI
from getpass import getpass
OPENAI_API_KEY = getpass("Enter your API key here :")

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [ ]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [ ]:
llm.invoke("Hello").content

In [ ]:
def add(a: int, b: int) -> int:
  "add a and b"
  return a + b

def multiply(a: int, b: int) -> int:
  "multiply a and b"
  return a * b

In [ ]:
tools = [search_duckduckgo, add, multiply]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
  return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
def search_duckduckgo(query: str):
  """Searches DuckDuckGo for the given query.""" # Added docstring here
  search = DuckDuckGoSearchRun()
  return search.invoke(query)

def add(a: int, b: int) -> int:
  """Adds a and b"""
  return a + b

def multiply(a: int, b: int) -> int:
  """Multiply a and b"""
  return a * b

tools = [search_duckduckgo, add, multiply]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
  return {"messages": [llm_with_tools.invoke(state["messages"])]}

from langgraph.prebuilt import ToolNode, tools_condition

graph_builder = StateGraph(State)

graph_builder.add_node("assistant", chatbot)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "assistant")
graph_builder.add_conditional_edges("assistant", tools_condition)
graph_builder.add_edge("tools", "assistant")

react_graph = graph_builder.compile()

In [ ]:
display(Image(react_graph.get_graph().draw_mermaid_png()))

In [ ]:
response = react_graph.invoke({"messages":[HumanMessage(content="What is the weather in Hydrabad. Multiply it by 2 and add 5.")]})

In [ ]:
response = react_graph.invoke({"messages":[HumanMessage(content="What is the weather in Bangalore. Multiply it by 2 and add 5.")]})

In [ ]:
print(response['messages'])

In [ ]:
response['messages']

# Conversational ChatBOT with Agentic AI (LangGraph)

In [ ]:
class State(TypedDict):
  messages: Annotated[list, add_messages]

In [ ]:
graph_builder = StateGraph(State)

In [ ]:
GROQ_API_KEY = getpass("Enter your GROQ API key here :")

In [ ]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
!pip install langchain_groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model='llama3-8b-8192')

In [ ]:
llm

In [ ]:
def chatbot(state:State):
  return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
from langgraph.graph import StateGraph, START, END

In [ ]:
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def langgraph_updates(user_input: str):
  # Use graph.stream instead of graph.system for streaming output
  for event in graph.stream({"messages": [{"role":"user", "content": user_input}]}):
    for value in event.values():
      if "messages" in value and len(value['messages']) > 0:
        # Corrected indexing to access the last message
        if value["messages"][-1].type == "ai":
          # Corrected indexing to access the content of the last message
          print("Assistant BOT :", value["messages"][-1].content)

while True:
  try:
    user_input = input("Enter your queries here what you want :")
    if user_input.lower() in ["bye","close","quit","terminate","exit","see you"]:
      print("Thank you for asking questions, bye and take care!")
      break

    langgraph_updates(user_input)
  except Exception as e:
    print(f"An error occured: {e}")
    # You might want to remove this hardcoded user_input and just print the error
    # or handle it differently based on your needs.
    # user_input = "Plan a weekend trip from Hydrabad to Delhi, including flights, hotels, and activities."
    # print("User : "+user_input)
    # langgraph_updates(user_input)
    break
